# Activation Functions
## Give a network more than one straight-line response

Imagine two switches controlling an indicator. The indicator should turn on when exactly one switch is on and turn off when both agree. This is XOR, the “exclusive or” rule. The two positive cases, (0, 1) and (1, 0), occupy opposite corners of a square; the negative cases occupy the other two corners. No straight line separates them. One weighted sum followed by a threshold therefore cannot express this rule on the raw switch inputs.

What should we change? Simply adding another weighted sum does not solve the problem. We need intermediate computations that respond differently in different regions of the input. Activation functions provide that change in behavior.

This lesson explains why nonlinear activations matter, how their shapes affect the signals passing through a network, and how hidden-layer choices differ from output-layer choices. We will work mostly with sigmoid, tanh, and ReLU, with brief extensions where they clarify the mechanism. The [code companion](03_Activation_Functions.ipynb) makes the curves, slopes, and XOR construction executable.

## Why stacking weighted sums is not enough

A neuron computes $z=wx+b$, followed by an output $a=\phi(z)$. Here $x$ is an input, $w$ is a weight, $b$ is a bias, $z$ is the pre-activation score, $\phi$ is the activation function, and $a$ is its output. With several inputs, the weighted sum uses a weight for each input.

Suppose we leave out the activation and compute two stages:

$$h=2x+1,\qquad y=3h-4.$$

$h$ is an intermediate value and $y$ is the final output. Substituting the first expression into the second gives

$$y=3(2x+1)-4=6x-1.$$

At $x=2$, the two-stage route gives $h=5$ and $y=11$. The collapsed route gives $6(2)-1=11$. It is still a single affine function: multiplication followed by an offset. This collapse also holds for multiple inputs and layers of matrix multiplications and biases.

Now insert ReLU between the stages:

$$h=\max(0,2x+1),\qquad y=3h-4.$$

The symbol $\max$ means choose the larger argument. If $x\leq-0.5$, the intermediate sum is nonpositive, so $h=0$ and $y=-4$. If $x>-0.5$, the output becomes $6x-1$. We now have two regions with different slopes. One straight line cannot describe both.

This is the useful meaning of **nonlinearity** here: the network can change its response pattern across the input space. ReLU is linear within each region, but its complete response has a bend. Several such units can combine bends into more complex functions. A nonlinear activation alone does not make every individual neuron's threshold boundary curved; its placement between learned transformations is what lets a multilayer model escape the collapse.

## First inspect what happens to actual scores

An activation normally acts on a neuron's score, not directly on the raw feature. For the elementwise functions below, each score is processed independently, and the tensor shape stays the same.

| Score $z$ | Identity | Sigmoid | Tanh | ReLU |
| ---: | ---: | ---: | ---: | ---: |
| -2 | -2 | 0.119 | -0.964 | 0 |
| 0 | 0 | 0.500 | 0.000 | 0 |
| 2 | 2 | 0.881 | 0.964 | 2 |

Identity leaves the score unchanged. Sigmoid compresses it between 0 and 1. Tanh compresses it between -1 and 1. ReLU clips negative values to zero and keeps positive values unchanged.

```text
Input batch       weighted sum + bias      activation       next computation
(B, d)       ──▶       (B, H)          ──▶    (B, H)    ──▶       ...
```

$B$ is the number of examples, $d$ is the number of input features, and $H$ is the number of output units in this layer. The activation changes values, not the number of examples or units. These familiar activation rules have no learned parameters of their own; their inputs depend on learned weights and biases.

## Sigmoid: a bounded, gradual response

The sigmoid function is

$$\sigma(z)=\frac{1}{1+e^{-z}}.$$

$\sigma$ names the sigmoid function, $z$ is its input score, and $e\approx2.718$ is the base of the natural exponential. At $z=2$, $e^{-2}\approx0.135$, so the output is $1/1.135\approx0.881$. At zero it is exactly 0.5. Large positive scores approach 1, while large negative scores approach 0.

This is useful when an output should represent a binary class probability estimate or a soft gate—a multiplier that controls how much of another value passes through. A sigmoid output still requires suitable training and evaluation before its probability interpretation is meaningful.

Look at the tails of the curve. Changing a score from 0 to 0.1 moves the output by about 0.025. Changing it from 8 to 8.1 moves the output by only about 0.000032. The response has nearly flattened. This is **saturation**: substantial input changes produce very small output changes.

To quantify the local response, a **derivative** measures how quickly output changes for a tiny change in input. Sigmoid's derivative is

$$\sigma'(z)=\sigma(z)\bigl(1-\sigma(z)\bigr).$$

The prime denotes derivative with respect to $z$. At zero the slope is $0.5(1-0.5)=0.25$. At $z=2$ it is about $0.881(0.119)=0.105$. Deep in either tail it approaches zero. These slopes help explain why repeated sigmoid hidden layers can weaken the learning signal.

## Tanh: keep a signed, bounded response

The hyperbolic tangent, usually called **tanh**, is

$$\tanh(z)=\frac{e^z-e^{-z}}{e^z+e^{-z}}.$$

$z$ is again the score and $e$ has the same meaning as before. For $z=1$, the numerator is approximately $2.718-0.368=2.350$ and the denominator is $2.718+0.368=3.086$, giving about 0.762. At $z=-1$, the result is about -0.762.

Tanh maps zero to zero and allows negative outputs. Its range is between -1 and 1, so it can express a bounded signed state. “Centered around zero” describes the function's symmetry; it does not guarantee that a particular batch of outputs has mean zero.

Its derivative is

$$\tanh'(z)=1-\tanh^2(z).$$

Here $\tanh^2(z)$ means square the tanh output. At zero the derivative is 1; near either saturated tail it approaches zero. Tanh therefore also has saturation-related limitations. It is not a universal cure for weak gradients.

## ReLU: retain the positive side

The rectified linear unit is

$$\operatorname{ReLU}(z)=\max(0,z).$$

At $z=-2$ its output is 0; at $z=2$ its output is 2. Away from zero, its slope is 0 for negative scores and 1 for positive scores. The positive side does not saturate. The output is also unbounded above, so ReLU does not by itself keep values small.

ReLU has a corner at zero, where the ordinary derivative does not exist. Software must choose a convention for gradient computations there; PyTorch uses zero. The presence of this corner does not prevent gradient-based training, because the function has useful derivatives away from the corner.

On the negative side, however, both the output and local slope are zero. If a unit's scores remain negative for all relevant training examples, the ordinary gradient signal through that unit can remain zero. This is the **dying ReLU** problem. One zero response on one example does not make a unit dead. Learning rate, initialization, and the data can affect whether units become persistently inactive.

A **leaky ReLU** keeps a small negative-side slope:

$$\phi(z)=\begin{cases}z,&z\geq0,\\ \alpha z,&z<0.\end{cases}$$

$\alpha$ is a chosen positive constant smaller than 1. With $\alpha=0.1$ and $z=-2$, the output is -0.2 and the local slope is 0.1. This provides a path for a nonzero gradient on negative inputs. It does not guarantee that the full model trains well.

For a smooth alternative, **GELU**, the Gaussian error linear unit, computes $z\Phi(z)$. Here $\Phi(z)$ is the probability that a standard normal random variable—mean zero, variance one—is at most $z$. It smoothly weights the input instead of hard-clipping the negative side. At $z=-1$, $\Phi(-1)\approx0.159$, giving output about -0.159; at $z=1$, the output is about 0.841. The [PyTorch GELU reference](https://docs.pytorch.org/docs/2.8/generated/torch.nn.GELU.html) defines the exact and approximate forms; we use the default form in the companion.

## A small nonlinear construction solves XOR

Return to the switches. Define two intermediate values:

$$h_1=\operatorname{ReLU}(x_1-x_2),\qquad h_2=\operatorname{ReLU}(x_2-x_1),\qquad s=h_1+h_2.$$

$x_1$ and $x_2$ are the switch inputs, $h_1$ and $h_2$ are hidden outputs, and $s$ is a final score. All biases are zero. The first unit responds when the first input is larger; the second responds when the second is larger. Adding the two measures the magnitude of their disagreement.

| $(x_1,x_2)$ | $x_1-x_2$ | $x_2-x_1$ | $h_1$ | $h_2$ | $s$ |
| --- | ---: | ---: | ---: | ---: | ---: |
| (0, 0) | 0 | 0 | 0 | 0 | 0 |
| (0, 1) | -1 | 1 | 0 | 1 | 1 |
| (1, 0) | 1 | -1 | 1 | 0 | 1 |
| (1, 1) | 0 | 0 | 0 | 0 | 0 |

Thresholding $s$ above 0.5 gives XOR exactly on these four inputs. Without the two ReLUs, the intermediate signed differences cancel: $(x_1-x_2)+(x_2-x_1)=0$ for every input. The activation prevents that cancellation by keeping different contributions in different regions.

For continuous inputs this construction computes $|x_1-x_2|$, absolute difference. It is not a calibrated probability. We hand-designed the weights; the demonstration establishes that the network **can express** XOR, not that any training procedure will discover these weights.

## Slopes influence how a change travels backward

A **gradient** collects derivatives that describe how an output or loss changes when inputs or parameters change. PyTorch's **autograd** records supported operations and computes such derivatives automatically.

For a scalar chain $h=f(x)$ and $y=g(h)$, the chain rule gives

$$\frac{dy}{dx}=\frac{dy}{dh}\frac{dh}{dx}.$$

$x$ is the original input, $h$ is the intermediate value, $y$ is the final output, and $f,g$ name the two functions. The equation says that the overall sensitivity is the product of the local sensitivities. If each of four stages contributes a slope of 0.25 and the intervening weights are 1, their product is $0.25^4=0.00390625$. Such repeated shrinking helps explain **vanishing gradients**, where some earlier computations receive very small signals for learning.

This example isolates one mechanism. Real networks also include weights, sums over multiple paths, and possibly shortcut connections. Large weights can amplify gradients; a ReLU path can still shut off or have shrinking weights. “ReLU always fixes vanishing gradients” would be incorrect.

The companion measures derivatives of simple functions and a repeated scalar chain. There is no loss or optimizer in those experiments. A derivative of output with respect to input is an illustration of sensitivity, not a complete training gradient analysis.

## Hidden activations and output activations do different jobs

Hidden activations help construct useful representations. Output transformations must match what the task asks the model to produce. For an unrestricted regression target, an identity output can produce any real value. For one binary label, a sigmoid can turn one raw score, or **logit**, into a class-1 probability estimate. For multiple labels that may all be true independently, apply a sigmoid to each label score.

When exactly one of several classes is correct, **softmax** converts a vector of scores into competing probabilities:

$$p_k=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}.$$

$K$ is the number of classes, $z_k$ is the score for class $k$, $j$ indexes every class in the denominator, and $p_k$ is the resulting probability for class $k$. The $K$ probabilities sum to 1. Unlike the elementwise functions above, each softmax output depends on all scores in that vector.

For scores $(2,1,0)$, the exponentials are approximately $(7.389,2.718,1)$, whose sum is 11.107. Dividing gives probabilities $(0.665,0.245,0.090)$. To avoid unnecessarily large exponentials, subtract the largest score before exponentiating. Scores $(0,-1,-2)$ give the same result because the common exponential factor cancels. Use the library implementation in practice.

With a batch of shape $(B,K)$, normalize across the class axis, not the batch axis. Softmax preserves the shape but couples the class values within each row.

In PyTorch, pass **raw logits** to `BCEWithLogitsLoss` for binary classification and to `CrossEntropyLoss` for the usual mutually exclusive multiclass classification setup. Apply sigmoid or softmax when you need displayed probabilities; do not apply them before these losses. The losses already handle the relevant operations stably. See the official [binary-loss](https://docs.pytorch.org/docs/2.8/generated/torch.nn.BCEWithLogitsLoss.html) and [multiclass-loss](https://docs.pytorch.org/docs/2.8/generated/torch.nn.CrossEntropyLoss.html) references.

## Put the choices in context

| Need | Useful mechanism | Limitation to remember |
| --- | --- | --- |
| A simple hidden nonlinearity | ReLU introduces bends and keeps positive-side slope 1. | Negative-side slope is zero; positive outputs are unbounded. |
| Some negative-side gradient | Leaky ReLU uses a small negative-side slope. | It does not solve every optimization problem. |
| A bounded signed state | Tanh preserves sign and bounds the output. | Both tails saturate. |
| A gate or binary probability output | Sigmoid maps a score between 0 and 1. | Hidden chains can saturate; probability reliability requires evaluation. |
| One choice among several classes | Softmax normalizes competing scores together. | It is inappropriate for independently true labels. |

Architecture, initialization, input scale, and the task also matter. There is no universally best activation. Compare choices on validation data when training a model rather than repeatedly using the test set to select one.

Common mistakes include leaving out all hidden nonlinearities, adding sigmoid twice, applying softmax over examples rather than classes, assuming sigmoid outputs are automatically calibrated probabilities, and calling any zero ReLU response a dead neuron. Also remember that the term “activation” can refer to either the function or its output values; context tells you which.

## Recap and self-check

Without nonlinearities, stacked affine layers collapse into one affine map. Activations change the response across input regions. Their slopes influence sensitivity, while their ranges help determine where they belong. Our two-unit ReLU construction expresses XOR, and softmax shows that not every activation processes scores independently.

- For scores $(-2,0,2)$, what does leaky ReLU with $\alpha=0.1$ output?
- Why can sigmoid output be close to 1 yet change very little when its input changes?
- Why does the XOR construction fail if both ReLUs become identity functions?
- For logits shaped $(8,3)$, which dimension should softmax normalize over?

Check: the leaky outputs are $(-0.2,0,2)$; sigmoid's positive tail saturates; the signed differences cancel; and softmax should normalize the three classes in each row (`dim=1`, equivalently `dim=-1`).